# IMGDS Experiment 1: Quantization x Sparsity Pareto Frontier
## 4 Bitwidths (32/16/8/4) x 7 Sparsity Levels (0/10/30/50/70/90/100%) = 28 Points

**Environment:** Docker Jupyter Kernel (http://127.0.0.1:8886)

**Project root:** `/home/cym/prj2/finn/notebooks/icl_thesis-master`

**Purpose:**
1. 28-point software quant + sparse simulation
2. Select HLS candidates for Pareto frontier
3. Generate v3 Pareto figures (Acc vs LUT, Acc vs Latency)
4. Prepare HLS project directories for candidates

**RULES:**
- NO retraining
- NO modifying src/transformer.py
- NO fabricating results
- HLS estimated != PYNQ measured
- Software simulation only for this notebook
- Real sparse = loop truncation (not mask x 0)


---
## Cell 1: Kernel Verification
---

In [ ]:
import sys, os, json, csv, time, warnings, copy
import numpy as np
from collections import OrderedDict

print(f'Python executable: {sys.executable}')
print(f'Python version: {sys.version}')

# CRITICAL: Verify Docker kernel
if 'anaconda' in sys.executable.lower() or 'conda' in sys.executable.lower():
    raise RuntimeError(
        f'ERROR: Using host conda Python ({sys.executable})!\n'
        'Please switch to Docker/Jupyter kernel:\n'
        '  Kernel -> Change Kernel -> Python 3 (ipykernel)'
    )
print('OK: Docker/Jupyter kernel confirmed.')

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    print(f'PyTorch {torch.__version__} available')
except ImportError as e:
    raise RuntimeError(f'PyTorch not available: {e}')


---
## Cell 2: Paths and Imports
---

In [ ]:
# Paths
PROJ = '/home/cym/prj2/finn/notebooks/icl_thesis-master'
EXP  = f'{PROJ}/experiments/imgds_linear_sparse'
FP   = f'{EXP}/final_paper_experiments'
TABLES = f'{FP}/tables'
FIGURES = f'{FP}/figures'
REPORTS = f'{FP}/reports'
STAGE2 = f'{EXP}/stage2_dense_hls_baselines'
CHECKPOINT_DIR = f'{EXP}/checkpoints'
ONNX_DIR = f'{EXP}/onnx'
OUTPUTS_DIR = f'{EXP}/outputs'

os.makedirs(TABLES, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)
os.makedirs(REPORTS, exist_ok=True)

sys.path.insert(0, f'{PROJ}/src')
from transformer import LinearUNSWAnomalyDetector

# Key constants (verified from HLS source)
D_MODEL = 16
SEQ_LEN = 16
INPUT_DIM = 64
FF_DIM = 32
N_CLASSES = 2

MODEL_CONFIG = {
    'input_dim': INPUT_DIM,
    'seq_len': SEQ_LEN,
    'd_model': D_MODEL,
    'dim_feedforward': FF_DIM,
    'num_layers': 1,
    'num_classes': N_CLASSES,
    'dropout': 0.0,
}

print('Paths configured.')
print(f'  PROJ: {PROJ}')
print(f'  D_MODEL={D_MODEL}, SEQ_LEN={SEQ_LEN}, INPUT_DIM={INPUT_DIM}')


---
## Cell 3: Experiment Matrix
---

In [ ]:
QUANT_BITS = [32, 16, 8, 4]
SPARSITY_PERCENT = [0, 10, 30, 50, 70, 90, 100]

QUANT_TYPE_MAP = {
    32: 'float32',
    16: 'ap_fixed<16,6>',
    8:  'ap_fixed<8,4>',
    4:  'ap_fixed<4,2>',
}

def sparsity_to_active_dim(sparsity_pct):
    if sparsity_pct == 100:
        return 0
    active = round(D_MODEL * (1 - sparsity_pct / 100.0))
    return max(active, 0)

def assign_stage(qbits, sparsity):
    if sparsity == 100 or (qbits == 32 and sparsity >= 70):
        return 'SOFTWARE_ONLY_ABLATION'
    if qbits == 32:
        return 'SOFTWARE_REFERENCE'
    if qbits == 4 and sparsity >= 50:
        return 'SOFTWARE_ABLATION'
    if sparsity >= 70:
        return 'SOFTWARE_ABLATION'
    return 'HLS_CANDIDATE'

experiments = []
for qbits in QUANT_BITS:
    for sp in SPARSITY_PERCENT:
        active_dim = sparsity_to_active_dim(sp)
        eid = f'e1_q{qbits}_s{sp}'
        exp = {
            'experiment_id': eid,
            'quant_bits': qbits,
            'quant_type': QUANT_TYPE_MAP[qbits],
            'sparsity_percent': sp,
            'active_dim': active_dim,
            'active_ratio': round(active_dim / D_MODEL, 4) if D_MODEL > 0 else 0.0,
            'stage': assign_stage(qbits, sp),
            'software_eval_status': 'PENDING',
            'hls_csim_status': 'PENDING',
            'hls_csynth_status': 'PENDING',
            'vivado_status': 'PENDING',
            'pynq_status': 'PENDING',
            'lut_reduction_target': '',
            'notes': '',
        }
        experiments.append(exp)

# Mark completed Q16_sp0
for exp in experiments:
    if exp['experiment_id'] == 'e1_q16_s0':
        exp['software_eval_status'] = 'DONE'
        exp['hls_csim_status'] = 'DONE'
        exp['hls_csynth_status'] = 'DONE'
        exp['vivado_status'] = 'DONE'
        exp['pynq_status'] = 'DONE'
        exp['stage'] = 'PYNQ_MEASURED'
        exp['notes'] = 'Completed: Q16 PARETO2-AXI; Vivado LUT=14004; PYNQ latency=2408.76us'
        break

print(f'Experiment matrix: {len(experiments)} points')
for e in experiments:
    print(f'{e["experiment_id"]:<18} Q={e["quant_bits"]} S={e["sparsity_percent"]:3d}% act_dim={e["active_dim"]:2d} stage={e["stage"]}')


---
## Cell 4: Save Matrix CSV
---

In [ ]:
matrix_csv = f'{TABLES}/experiment1_quant_sparsity_matrix.csv'
fieldnames = ['experiment_id','quant_bits','quant_type','sparsity_percent',
              'active_dim','active_ratio','stage','software_eval_status',
              'hls_csim_status','hls_csynth_status','vivado_status',
              'pynq_status','lut_reduction_target','notes']

with open(matrix_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(experiments)

print(f'Saved: {matrix_csv}')
print(f'Total: {len(experiments)} points')


---
## Cell 5: Load Trained Checkpoint and Test Data
---

**NO RETRAINING. NO MODIFICATION to src/transformer.py.**

In [ ]:
CKPT_PATH = f'{CHECKPOINT_DIR}/best_linear_imgds_dense_r32_p8.pt'
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f'Checkpoint not found: {CKPT_PATH}')

ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
print(f'Checkpoint keys: {list(ckpt.keys())}')

if 'state_dict' in ckpt:
    state_dict = ckpt['state_dict']
elif 'model_state_dict' in ckpt:
    state_dict = ckpt['model_state_dict']
else:
    state_dict = ckpt

total_params = sum(v.numel() for v in state_dict.values())
print(f'Parameters: {total_params}')
for k, v in state_dict.items():
    print(f'  {k:<55s} shape={str(v.shape):<20s}')

TEST_NPZ = f'{OUTPUTS_DIR}/imgds_r32_p8_test.npz'
test_data = np.load(TEST_NPZ)
X_test = test_data['X']
y_test = test_data['y']
print(f'Test data: X={X_test.shape}, y={y_test.shape}')
print(f'Labels: {np.bincount(y_test)}')


---
## Cell 6: Fake Quantization Functions
---

ap_fixed<16,6>: 16-bit, 6 int bits. ap_fixed<8,4>: 8-bit, 4 int bits. ap_fixed<4,2>: 4-bit, 2 int bits.

In [ ]:
def fake_quantize_tensor(w, total_bits, int_bits):
    if total_bits == 32:
        return w.clone()
    frac_bits = total_bits - int_bits
    max_val = 2**(int_bits - 1) - 2**(-frac_bits)
    min_val = -2**(int_bits - 1)
    scale = 2**frac_bits
    w_scaled = w * scale
    w_rounded = torch.round(w_scaled)
    w_clamped = torch.clamp(w_rounded, min_val * scale, max_val * scale)
    return w_clamped / scale

def fake_quantize_state_dict(state_dict, total_bits):
    if total_bits == 32:
        return OrderedDict((k, v.clone()) for k, v in state_dict.items())
    quantized = OrderedDict()
    for k, v in state_dict.items():
        is_weight = 'weight' in k and v.ndim >= 2
        is_norm = 'norm' in k.lower()
        if is_weight and not is_norm:
            if total_bits == 16:
                quantized[k] = fake_quantize_tensor(v, 16, 6)
            elif total_bits == 8:
                quantized[k] = fake_quantize_tensor(v, 8, 4)
            elif total_bits == 4:
                quantized[k] = fake_quantize_tensor(v, 4, 2)
            else:
                quantized[k] = v.clone()
        else:
            quantized[k] = v.clone()
    return quantized

# Test
test_w = state_dict['backbone.input_projection.weight']
print(f'Original: min={test_w.min():.6f}, max={test_w.max():.6f}')
for bits in [16, 8, 4]:
    qw = fake_quantize_tensor(test_w, bits, {16:6, 8:4, 4:2}[bits])
    err = (qw - test_w).abs()
    print(f'  Q{bits}: max_err={err.max():.6f}, mean_err={err.mean():.8f}')
print('Quantization functions ready.')


---
## Cell 7: Channel Importance Analysis and Sparsity Application
---

**IMPORTANT:** Sparsity here means reducing active D_MODEL channels. In software we zero out weights for inactive channels (simulating truncated HLS loops).

In HLS, the sparse version uses a truncated loop:
```cpp
for (int ii = 0; ii < ACTIVE_DIM; ii++) {
    int c = active_idx[ii];
    sum += q[c] * k[c];
}
```

**NOT:** `sum += q[c] * k[c] * mask[c];` (which still synthesizes full compute)

Channel importance: L1 norm across all attention and FF weight matrices.

In [ ]:
def compute_channel_importance(state_dict):
    importance = np.zeros(D_MODEL)
    # Attention weights: query, key, value, output [D_MODEL, D_MODEL]
    for prefix in ['backbone.layers.0.attention.query',
                   'backbone.layers.0.attention.key',
                   'backbone.layers.0.attention.value',
                   'backbone.layers.0.attention.output']:
        w = state_dict[f'{prefix}.weight'].numpy()
        importance += np.abs(w).sum(axis=1)  # row norm
        importance += np.abs(w).sum(axis=0)  # col norm
    # Feedforward: fc0 [32,16] -> col, fc3 [16,32] -> row
    w = state_dict['backbone.layers.0.feedforward.0.weight'].numpy()
    importance += np.abs(w).sum(axis=1)[:D_MODEL]
    w = state_dict['backbone.layers.0.feedforward.3.weight'].numpy()
    importance += np.abs(w).sum(axis=0)[:D_MODEL]
    # Classifier [2, 16]
    w = state_dict['backbone.classifier.weight'].numpy()
    importance += np.abs(w).sum(axis=0)
    # Input projection [16, 64]
    w = state_dict['backbone.input_projection.weight'].numpy()
    importance += np.abs(w).sum(axis=1)
    return importance

def select_active_channels(importance, active_dim):
    if active_dim >= D_MODEL:
        return list(range(D_MODEL))
    if active_dim <= 0:
        return []
    sorted_idx = np.argsort(importance)[::-1]
    return sorted(sorted_idx[:active_dim].tolist())

def apply_channel_sparsity(state_dict, active_channels):
    if len(active_channels) == D_MODEL:
        return state_dict
    sparse_sd = OrderedDict()
    inactive_mask = np.ones(D_MODEL, dtype=np.float32)
    for c in range(D_MODEL):
        if c not in active_channels:
            inactive_mask[c] = 0.0
    
    for k, v in state_dict.items():
        v_np = v.numpy().copy()
        shape = v_np.shape
        
        if 'input_projection' in k and 'weight' in k and shape == (D_MODEL, INPUT_DIM):
            v_np = v_np * inactive_mask[:, None]
        elif 'input_projection' in k and 'bias' in k and shape == (D_MODEL,):
            v_np = v_np * inactive_mask
        elif 'attention' in k and 'weight' in k and shape == (D_MODEL, D_MODEL):
            mask_2d = inactive_mask[:, None] * inactive_mask[None, :]
            v_np = v_np * mask_2d
        elif 'attention' in k and 'bias' in k and shape == (D_MODEL,):
            v_np = v_np * inactive_mask
        elif 'feedforward.0' in k and 'weight' in k and shape == (FF_DIM, D_MODEL):
            v_np = v_np * inactive_mask[None, :]
        elif 'feedforward.3' in k and 'weight' in k and shape == (D_MODEL, FF_DIM):
            v_np = v_np * inactive_mask[:, None]
        elif 'feedforward.3' in k and 'bias' in k and shape == (D_MODEL,):
            v_np = v_np * inactive_mask
        elif 'classifier' in k and 'weight' in k and shape == (N_CLASSES, D_MODEL):
            v_np = v_np * inactive_mask[None, :]
        elif 'position_embedding' in k and len(shape) == 3 and shape[2] == D_MODEL:
            v_np = v_np * inactive_mask[None, None, :]
        
        sparse_sd[k] = torch.from_numpy(v_np)
    return sparse_sd

# Compute channel importance
channel_importance = compute_channel_importance(state_dict)
sorted_idx = np.argsort(channel_importance)[::-1]
print('Channel importance (highest first):')
for rank, ch in enumerate(sorted_idx):
    print(f'  Rank {rank+1}: ch={ch} importance={channel_importance[ch]:.4f}')

# Cache active channel sets
active_channels_cache = {}
for sp in SPARSITY_PERCENT:
    active_dim = sparsity_to_active_dim(sp)
    active_channels_cache[sp] = select_active_channels(channel_importance, active_dim)
    print(f'  S={sp:3d}% -> active_dim={active_dim:2d} -> channels={active_channels_cache[sp]}')


---
## Cell 8: Create Model and Define Inference
---

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def create_model_with_weights(state_dict_to_load):
    model = LinearUNSWAnomalyDetector(**MODEL_CONFIG)
    model.load_state_dict(state_dict_to_load, strict=True)
    model.eval()
    return model

def run_inference(model, X, batch_size=512):
    all_logits = []
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            batch = torch.from_numpy(X[i:i+batch_size]).float()
            logits = model(batch)
            preds = torch.argmax(logits, dim=1)
            all_logits.append(logits.numpy())
            all_preds.append(preds.numpy())
    return np.concatenate(all_logits, axis=0), np.concatenate(all_preds, axis=0)

def compute_metrics(y_true, y_pred, logits, ref_logits=None):
    metrics = {}
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
    metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
    metrics['f1'] = f1_score(y_true, y_pred, zero_division=0)
    probs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
    if len(np.unique(y_true)) > 1:
        metrics['auc'] = roc_auc_score(y_true, probs[:, 1])
    else:
        metrics['auc'] = 0.5
    if ref_logits is not None:
        ref_preds = np.argmax(ref_logits, axis=1)
        metrics['prediction_match_rate_vs_q32'] = (y_pred == ref_preds).mean()
        logit_errors = np.abs(logits - ref_logits)
        metrics['max_logit_error_vs_q32'] = logit_errors.max()
        metrics['mean_logit_error_vs_q32'] = logit_errors.mean()
    metrics['num_samples'] = len(y_true)
    return metrics

# Baseline Q32
print('Running Q32 baseline...')
model_q32_s0 = create_model_with_weights(state_dict)
logits_q32, preds_q32 = run_inference(model_q32_s0, X_test)
metrics_q32 = compute_metrics(y_test, preds_q32, logits_q32)
print(f'  Q32 baseline: acc={metrics_q32["accuracy"]:.4f}, f1={metrics_q32["f1"]:.4f}, auc={metrics_q32["auc"]:.4f}')


---
## Cell 9: Run All 28 Software Simulations
---

**This is the main computation.** For each (quant_bits, sparsity) combination:
1. Fake quantize weights
2. Apply channel sparsity
3. Run inference on 3,444 test samples
4. Compute metrics vs Q32 reference

In [ ]:
import time as time_module

print('=' * 80)
print('Running 28 Software Simulations')
print('=' * 80)

# Pre-compute active channel sets
channel_importance = compute_channel_importance(state_dict)
for sp in SPARSITY_PERCENT:
    active_dim = sparsity_to_active_dim(sp)
    active_channels_cache[sp] = select_active_channels(channel_importance, active_dim)

sw_results = []
start_time = time_module.time()

for exp in experiments:
    qbits = exp['quant_bits']
    sp = exp['sparsity_percent']
    eid = exp['experiment_id']
    
    t0 = time_module.time()
    label = f'[{eid}] Q={qbits}bit S={sp}%'
    
    try:
        # Step 1: Fake quantize
        q_sd = fake_quantize_state_dict(state_dict, qbits)
        # Step 2: Channel sparsity
        active_ch = active_channels_cache[sp]
        qs_sd = apply_channel_sparsity(q_sd, active_ch)
        # Step 3: Inference
        model = create_model_with_weights(qs_sd)
        logits, preds = run_inference(model, X_test)
        # Step 4: Metrics
        metrics = compute_metrics(y_test, preds, logits, ref_logits=logits_q32)
        
        result = {
            'experiment_id': eid,
            'quant_bits': qbits,
            'quant_type': exp['quant_type'],
            'sparsity_percent': sp,
            'active_dim': exp['active_dim'],
            'accuracy': round(metrics['accuracy'], 6),
            'precision': round(metrics['precision'], 6),
            'recall': round(metrics['recall'], 6),
            'f1': round(metrics['f1'], 6),
            'auc': round(metrics['auc'], 6),
            'prediction_match_rate_vs_q32': round(metrics.get('prediction_match_rate_vs_q32', 1.0), 6),
            'max_logit_error_vs_q32': round(metrics.get('max_logit_error_vs_q32', 0.0), 6),
            'mean_logit_error_vs_q32': round(metrics.get('mean_logit_error_vs_q32', 0.0), 6),
            'num_samples': metrics['num_samples'],
            'status': 'SUCCESS',
            'notes': '',
        }
        sw_results.append(result)
        dt = time_module.time() - t0
        print(f'{label}: acc={metrics["accuracy"]:.4f} match={metrics["prediction_match_rate_vs_q32"]:.4f} ({dt:.1f}s)')
        
    except Exception as ex:
        dt = time_module.time() - t0
        print(f'{label}: FAILED - {ex} ({dt:.1f}s)')
        result = {
            'experiment_id': eid,
            'quant_bits': qbits,
            'quant_type': exp['quant_type'],
            'sparsity_percent': sp,
            'active_dim': exp['active_dim'],
            'accuracy': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'auc': 0.5,
            'prediction_match_rate_vs_q32': 0.0,
            'max_logit_error_vs_q32': float('inf'),
            'mean_logit_error_vs_q32': float('inf'),
            'num_samples': 0,
            'status': f'FAILED: {ex}',
            'notes': str(ex)[:200],
        }
        sw_results.append(result)

total_time = time_module.time() - start_time
n_ok = sum(1 for r in sw_results if r['status'] == 'SUCCESS')
print(f'\nDone: {n_ok}/{len(experiments)} in {total_time:.1f}s')


---
## Cell 10: Save Software Results
---

In [ ]:
results_csv = f'{TABLES}/experiment1_software_quant_sparsity_results.csv'
fieldnames = ['experiment_id','quant_bits','quant_type','sparsity_percent',
              'active_dim','accuracy','precision','recall','f1','auc',
              'prediction_match_rate_vs_q32','max_logit_error_vs_q32',
              'mean_logit_error_vs_q32','num_samples','status','notes']

with open(results_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(sw_results)

print(f'Saved: {results_csv}')

# Print summary
print(f'\n{"ID":<18} {"Q":<4} {"S%":<5} {"ActD":<5} {"Acc":<8} {"Match":<8} {"MaxErr":<10} {"Status"}')
print('-' * 80)
for r in sw_results:
    if r['status'] == 'SUCCESS':
        print(f'{r["experiment_id"]:<18} {r["quant_bits"]:<4} {r["sparsity_percent"]:<5} {r["active_dim"]:<5} '
              f'{r["accuracy"]:<8.4f} {r["prediction_match_rate_vs_q32"]:<8.4f} '
              f'{r["max_logit_error_vs_q32"]:<10.6f} {r["status"]}')
    else:
        print(f'{r["experiment_id"]:<18} {r["quant_bits"]:<4} {r["sparsity_percent"]:<5} {r["active_dim"]:<5} '
              f'{"FAIL":<8} {"N/A":<8} {"N/A":<10} {r["status"]}')


---
## Cell 11: HLS Candidate Selection
---

**Rules:**
1. accuracy >= 92% OR prediction_match_rate >= 95%
2. At least 1 rep per quant bit
3. 100% sparsity = SOFTWARE_ONLY
4. Q32 = SOFTWARE_REFERENCE (no HLS required)
5. Priority: low LUT + high accuracy

In [ ]:
ACC_THRESHOLD = 0.92
MATCH_THRESHOLD = 0.95
BASE_HLS_LUT = 25402
BASE_VIVADO_LUT = 14004

hls_candidates = []

for r in sw_results:
    if r['status'] != 'SUCCESS':
        continue
    qbits = r['quant_bits']
    sp = r['sparsity_percent']
    acc = r['accuracy']
    match = r['prediction_match_rate_vs_q32']
    
    if sp == 100:
        r['hls_candidate'] = False
        r['hls_reason'] = 'Zero-attention baseline'
        continue
    if qbits == 32:
        r['hls_candidate'] = False
        r['hls_reason'] = 'Float32 reference'
        continue
    
    if acc >= ACC_THRESHOLD or match >= MATCH_THRESHOLD:
        r['hls_candidate'] = True
        r['hls_reason'] = f'acc={acc:.4f}>={ACC_THRESHOLD} or match={match:.4f}>={MATCH_THRESHOLD}'
        hls_candidates.append(r)
    else:
        r['hls_candidate'] = False
        r['hls_reason'] = f'Below threshold'

# Ensure per-quant representation
for qbits in [16, 8, 4]:
    existing = [c for c in hls_candidates if c['quant_bits'] == qbits]
    if not existing:
        q_pts = [r for r in sw_results if r['quant_bits'] == qbits and r['status'] == 'SUCCESS' and r['sparsity_percent'] < 100]
        if q_pts:
            best = max(q_pts, key=lambda x: x['accuracy'])
            best['hls_candidate'] = True
            best['hls_reason'] = f'Forced Q{qbits} rep (best acc={best["accuracy"]:.4f})'
            hls_candidates.append(best)

# Save selection CSV
hls_csv = f'{TABLES}/experiment1_hls_candidate_selection.csv'
hls_fieldnames = ['experiment_id','quant_bits','quant_type','sparsity_percent',
                  'active_dim','accuracy','prediction_match_rate_vs_q32',
                  'hls_candidate','hls_reason','lut_reduction_potential','priority_rank']

with open(hls_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=hls_fieldnames, extrasaction='ignore')
    writer.writeheader()
    
    all_with_flag = []
    for r in sw_results:
        if r['status'] != 'SUCCESS':
            continue
        is_candidate = r.get('hls_candidate', False)
        reason = r.get('hls_reason', '')
        bit_factor = r['quant_bits'] / 16.0 if r['quant_bits'] < 32 else 1.0
        dim_factor = max(r['active_dim'], 1) / D_MODEL
        est_hls_lut = int(BASE_HLS_LUT * bit_factor * dim_factor)
        est_vivado_lut = int(BASE_VIVADO_LUT * bit_factor * dim_factor)
        priority = r['accuracy'] / (est_vivado_lut + 1) * 10000 if is_candidate else 999
        
        row = {
            'experiment_id': r['experiment_id'],
            'quant_bits': r['quant_bits'],
            'quant_type': r['quant_type'],
            'sparsity_percent': r['sparsity_percent'],
            'active_dim': r['active_dim'],
            'accuracy': r['accuracy'],
            'prediction_match_rate_vs_q32': r['prediction_match_rate_vs_q32'],
            'hls_candidate': is_candidate,
            'hls_reason': reason,
            'lut_reduction_potential': f'HLS est: ~{est_hls_lut} LUT, Vivado est: ~{est_vivado_lut} LUT',
            'priority_rank': round(priority, 4) if is_candidate else 999,
        }
        all_with_flag.append(row)
    
    all_with_flag.sort(key=lambda x: x['priority_rank'], reverse=True)
    writer.writerows(all_with_flag)

print(f'Saved: {hls_csv}')
print(f'HLS Candidates ({len(hls_candidates)}):')
for c in sorted(hls_candidates, key=lambda x: x['accuracy'], reverse=True):
    print(f'  {c["experiment_id"]:<18} Q={c["quant_bits"]} S={c["sparsity_percent"]}% '
          f'acc={c["accuracy"]:.4f} match={c["prediction_match_rate_vs_q32"]:.4f}')


---
## Cell 12: Prepare Stage 3 HLS Directories
---

Copy from PARETO2-AXI baseline. Generate config headers. **DOES NOT run HLS.**

In [ ]:
import shutil

STAGE3_DIR = f'{EXP}/stage3_quant_sparsity_hls'
os.makedirs(STAGE3_DIR, exist_ok=True)
BASELINE_SRC = f'{STAGE2}/q16_s0_pareto2_axi/hls/src'

def generate_quant_sparse_config(qbits, sparsity, active_dim):
    lines = [
        '// quant_sparse_config.h - Auto-generated',
        f'#ifndef QUANT_SPARSE_CONFIG_H',
        f'#define QUANT_SPARSE_CONFIG_H',
        '',
        f'#define QUANT_BITS       {qbits}',
        f'#define SPARSITY_PERCENT {sparsity}',
        f'#define ACTIVE_DIM       {active_dim}',
        f'#define D_MODEL          16',
        f'#define SEQ_LEN          16',
        f'#define INPUT_DIM        64',
        f'#define FF_DIM           32',
        f'#define N_CLASSES        2',
        '',
    ]
    if qbits == 32:
        lines += ['typedef float data_t;', 'typedef float acc_t;']
    elif qbits == 16:
        lines += ['#include <ap_fixed.h>', 'typedef ap_fixed<16,6> data_t;', 'typedef ap_fixed<20,8> acc_t;']
    elif qbits == 8:
        lines += ['#include <ap_fixed.h>', 'typedef ap_fixed<8,4> data_t;', 'typedef ap_fixed<20,8> acc_t;']
    elif qbits == 4:
        lines += ['#include <ap_fixed.h>', 'typedef ap_fixed<4,2> data_t;', 'typedef ap_fixed<20,8> acc_t;']
    lines += ['', f'#endif', '']
    return '\n'.join(lines)

def generate_active_channels_h(active_channels):
    ch_list = ', '.join(str(c) for c in active_channels)
    n = len(active_channels)
    lines = [
        '// active_channels.h - Auto-generated',
        '// Channel selection based on L1 norm importance',
        f'#ifndef ACTIVE_CHANNELS_H',
        f'#define ACTIVE_CHANNELS_H',
        '',
        f'#define ACTIVE_DIM {n}',
        f'static const int active_idx[ACTIVE_DIM] = {{{ch_list}}};',
        '',
        f'#endif',
        '',
    ]
    return '\n'.join(lines)

prepared_dirs = []
for c in hls_candidates:
    eid = c['experiment_id']
    qbits = c['quant_bits']
    sp = c['sparsity_percent']
    active_dim = c['active_dim']
    dir_name = f'q{qbits}_s{sp}_pareto_axi'
    dir_path = f'{STAGE3_DIR}/{dir_name}'
    
    print(f'{dir_name} ...')
    os.makedirs(f'{dir_path}/hls/src', exist_ok=True)
    os.makedirs(f'{dir_path}/reports', exist_ok=True)
    os.makedirs(f'{dir_path}/logs', exist_ok=True)
    
    # Copy baseline HLS source
    for src_file in ['imgds_linear_dense_q16.cpp', 'imgds_linear_dense_q16.h', 'testbench.cpp']:
        src_path = f'{BASELINE_SRC}/{src_file}'
        if os.path.exists(src_path):
            shutil.copy2(src_path, f'{dir_path}/hls/src/{src_file}')
    
    # Generate config
    with open(f'{dir_path}/hls/src/quant_sparse_config.h', 'w') as f:
        f.write(generate_quant_sparse_config(qbits, sp, active_dim))
    
    # Generate active channels
    active_ch = active_channels_cache[sp]
    with open(f'{dir_path}/hls/src/active_channels.h', 'w') as f:
        f.write(generate_active_channels_h(active_ch))
    
    # Copy data files
    for data_file in ['eval_samples_q16.h', 'hls_params_q16.h']:
        src_path = f'{BASELINE_SRC}/{data_file}'
        if os.path.exists(src_path):
            shutil.copy2(src_path, f'{dir_path}/hls/src/{data_file}')
    
    # README
    readme = f'''# {dir_name}
Experiment 1: Q{qbits} quantization, {sp}% sparse
Quant type: {QUANT_TYPE_MAP[qbits]}
Active channels: {active_dim}/{D_MODEL} = {active_ch}
SW accuracy: {c["accuracy"]:.4f}
SW match rate: {c["prediction_match_rate_vs_q32"]:.4f}

Status: DIRECTORY_PREPARED (HLS not yet run)
Source: Copied from q16_s0_pareto2_axi baseline
'''
    with open(f'{dir_path}/README.md', 'w') as f:
        f.write(readme)
    
    prepared_dirs.append(dir_path)

print(f'\nPrepared {len(prepared_dirs)} HLS directories under: {STAGE3_DIR}/')
for d in prepared_dirs:
    print(f'  {os.path.basename(d)}/')


---
## Cell 13: Generate v3 Pareto Figures
---

**Figure 1 v3: Accuracy vs LUT**
**Figure 2 v3: Accuracy vs Latency**

Legend:
- BLUE squares: ViT4Mal literature
- RED hollow circles: Ours HLS estimated
- RED solid stars: Ours PYNQ measured (Q16 S0 only)
- GRAY hollow circles: Ours SW simulation (estimated LUT/latency)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300, 'font.size': 10})

# ViT4Mal literature points
vit4mal_pts = [
    ('1enc 2head (fastest)', 92.41, 16545, 810000),
    ('1enc 4head', 92.47, 16029, 870000),
    ('2enc 2head', 93.86, 16219, 1450000),
    ('2enc 4head (rec.)', 93.89, 16079, 1480000),
    ('4enc 2head', 93.16, 16200, 2810000),
    ('4enc 4head', 92.16, 16135, 2820000),
]

# Ours HLS estimated (existing)
hls_est = [
    ('Q16 naive', 94.50, 180835, 25.66),
    ('Q16 PARETO1', 94.50, 146862, 62.08),
    ('Q16 PARETO2', 94.50, 22056, 740.84),
    ('Q16 PARETO2-AXI', 99.0, 25402, 756.27),
]

# Ours PYNQ measured
pynq_meas = [
    ('Q16 S0 PARETO2-AXI', 99.0, 14004, 2408.76),
]

# Ours SW simulation points (estimated LUT/latency)
sw_pts = []
for r in sw_results:
    if r['status'] != 'SUCCESS' or r['accuracy'] <= 0:
        continue
    if r['experiment_id'] in ['e1_q32_s0', 'e1_q16_s0']:
        continue
    qbits = r['quant_bits']
    sp = r['sparsity_percent']
    acc_pct = r['accuracy'] * 100
    bit_factor = qbits / 16.0 if qbits < 32 else 1.0
    dim_factor = max(r['active_dim'], 1) / D_MODEL
    est_lut = int(BASE_VIVADO_LUT * bit_factor * dim_factor)
    est_lat = 2408.76 * bit_factor * dim_factor
    label = f'Q{qbits} S{sp}%'
    sw_pts.append((label, acc_pct, est_lut, est_lat, qbits, sp))

print(f'SW points for plot: {len(sw_pts)}')
print(f'ViT4Mal: {len(vit4mal_pts)}, HLS est: {len(hls_est)}, PYNQ: {len(pynq_meas)}')


In [ ]:
# ===== FIGURE 1 v3: Accuracy vs LUT =====
fig1, ax1 = plt.subplots(figsize=(12, 7))

# ViT4Mal
for i, (label, acc, lut, lat) in enumerate(vit4mal_pts):
    ax1.scatter(lut, acc, c='blue', marker='s', s=100, zorder=5,
               edgecolors='navy', linewidth=0.8,
               label='ViT4Mal (literature)' if i == 0 else '')
    ax1.annotate(label, (lut, acc), textcoords='offset points', xytext=(8, -8),
                fontsize=7, alpha=0.8, color='navy')

# Ours HLS estimated
for i, (label, acc, lut, lat) in enumerate(hls_est):
    ax1.scatter(lut, acc, c='none', marker='o', s=120, zorder=6,
               edgecolors='red', linewidth=1.5,
               label='Ours HLS estimated' if i == 0 else '')
    ax1.annotate(label, (lut, acc), textcoords='offset points', xytext=(10, 5),
                fontsize=7, color='darkred')

# Ours PYNQ measured
for i, (label, acc, lut, lat) in enumerate(pynq_meas):
    ax1.scatter(lut, acc, c='red', marker='*', s=300, zorder=8,
               edgecolors='darkred', linewidth=1.5,
               label='Ours PYNQ-Z2 measured' if i == 0 else '')
    ax1.annotate(f'{label}\nLUT={lut} Acc={acc}% PYNQ',
                (lut, acc), textcoords='offset points', xytext=(15, -5),
                fontsize=8, color='darkred', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

# Ours SW sim (gray, hollow)
sw_good = [p for p in sw_pts if p[1] >= 85]
for i, (label, acc, est_lut, est_lat, qbits, sp) in enumerate(sw_good):
    ax1.scatter(est_lut, acc, c='none', marker='o', s=60, zorder=4,
               edgecolors='gray', linewidth=0.8, alpha=0.5,
               label='Ours SW sim (est.)' if i == 0 else '')

# Reference lines
ax1.axvline(x=53200, color='orange', linestyle='--', linewidth=1.5,
           label='PYNQ-Z2 LUT limit (53,200)', alpha=0.7)
ax1.axvspan(0, 14004, alpha=0.08, color='green', label='Target: LUT <14,004')
ax1.axhline(y=92.41, color='green', linestyle=':', linewidth=1, alpha=0.5)
ax1.axhline(y=93.89, color='darkgreen', linestyle=':', linewidth=1, alpha=0.5)

ax1.annotate('Target zone:\nLUT < 14,004\nAcc > 92.41%',
            xy=(4000, 96.5), fontsize=9, color='green', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

ax1.set_xlabel('LUT Count')
ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Fig 1 (v3): Accuracy vs LUT - Quant x Sparsity Pareto Frontier')
ax1.legend(loc='lower right', fontsize=7, ncol=2)
ax1.set_xlim(-2000, 60000)
ax1.set_ylim(84, 101)
ax1.grid(True, alpha=0.2)

ax1.text(0.98, 0.02,
        'Gray = SW sim (LUT estimated). Red star = PYNQ measured (only Q16 S0).',
        transform=ax1.transAxes, fontsize=7, ha='right', va='bottom',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
fig1_path = f'{FIGURES}/fig1_acc_vs_lut_v3_quant_sparsity_pareto.png'
fig1.savefig(fig1_path, dpi=300, bbox_inches='tight')
print(f'Saved: {fig1_path}')
plt.show()


In [ ]:
# ===== FIGURE 2 v3: Accuracy vs Latency =====
fig2, ax2 = plt.subplots(figsize=(12, 7))

# ViT4Mal
for i, (label, acc, lut, lat) in enumerate(vit4mal_pts):
    ax2.scatter(lat, acc, c='blue', marker='s', s=100, zorder=5,
               edgecolors='navy', linewidth=0.8,
               label='ViT4Mal' if i == 0 else '')
    ax2.annotate(f'{label}\n{lat/1000:.2f}ms', (lat, acc), textcoords='offset points',
                xytext=(10, -10), fontsize=7, alpha=0.8, color='navy')

# Ours HLS estimated
for i, (label, acc, lut, lat) in enumerate(hls_est):
    ax2.scatter(lat, acc, c='none', marker='o', s=120, zorder=6,
               edgecolors='red', linewidth=1.5,
               label='Ours HLS est.' if i == 0 else '')
    ax2.annotate(f'{label}\n{lat:.1f}us', (lat, acc), textcoords='offset points',
                xytext=(10, 5), fontsize=7, color='darkred')

# Ours PYNQ measured
for i, (label, acc, lut, lat) in enumerate(pynq_meas):
    ax2.scatter(lat, acc, c='red', marker='*', s=300, zorder=8,
               edgecolors='darkred', linewidth=1.5,
               label='Ours PYNQ-Z2 measured')
    ax2.annotate(f'{label}\n{lat:.1f}us = {lat/1000:.2f}ms\n336x vs ViT4Mal',
                (lat, acc), textcoords='offset points', xytext=(15, -15),
                fontsize=8, color='darkred', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

# Ours SW sim
for i, (label, acc, est_lut, est_lat, qbits, sp) in enumerate(sw_good):
    ax2.scatter(est_lat, acc, c='none', marker='o', s=60, zorder=4,
               edgecolors='gray', linewidth=0.8, alpha=0.5,
               label='Ours SW sim (est.)' if i == 0 else '')

ax2.set_xscale('log')
ax2.set_xlabel('Latency (us, log scale)')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Fig 2 (v3): Accuracy vs Latency - Quant x Sparsity Pareto Frontier')
ax2.legend(loc='lower left', fontsize=7, ncol=2)
ax2.set_ylim(84, 101)
ax2.grid(True, alpha=0.2, which='both')

ax2.text(0.98, 0.02,
        'Gray = SW sim (latency estimated). Red star = PYNQ measured (only Q16 S0).',
        transform=ax2.transAxes, fontsize=7, ha='right', va='bottom',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
fig2_path = f'{FIGURES}/fig2_acc_vs_latency_v3_quant_sparsity_pareto.png'
fig2.savefig(fig2_path, dpi=300, bbox_inches='tight')
print(f'Saved: {fig2_path}')
plt.show()


---
## Cell 14: Best Pareto Frontier Candidates
---

In [ ]:
print('=' * 80)
print('BEST PARETO CANDIDATES - Ranked by LUT Reduction')
print('=' * 80)

print(f'\nViT4Mal fastest:  LUT=16,545  Acc=92.41%  Lat=810,000us')
print(f'ViT4Mal rec.:     LUT=16,079  Acc=93.89%  Lat=1,480,000us')
print(f'Ours Q16 S0 PYNQ: LUT=14,004  Acc=99.0%   Lat=2,408.76us (MEASURED)')

print(f'\n--- Points with est. LUT < 14,004 ---')
for r in sw_results:
    if r['status'] != 'SUCCESS':
        continue
    qbits = r['quant_bits']
    sp = r['sparsity_percent']
    bit_factor = qbits / 16.0 if qbits < 32 else 1.0
    dim_factor = max(r['active_dim'], 1) / D_MODEL
    est_lut = int(BASE_VIVADO_LUT * bit_factor * dim_factor)
    acc_pct = r['accuracy'] * 100
    if est_lut < 14004:
        star = '*' if acc_pct >= 92.41 else ' '
        print(f'  {star} {r["experiment_id"]:<18} Q={qbits} S={sp}% '
              f'est_LUT={est_lut:<6} acc={acc_pct:.2f}% match={r["prediction_match_rate_vs_q32"]:.4f}')

print(f'\n--- Points with est. LUT between 14,004 and 16,545 ---')
for r in sw_results:
    if r['status'] != 'SUCCESS':
        continue
    qbits = r['quant_bits']
    sp = r['sparsity_percent']
    bit_factor = qbits / 16.0 if qbits < 32 else 1.0
    dim_factor = max(r['active_dim'], 1) / D_MODEL
    est_lut = int(BASE_VIVADO_LUT * bit_factor * dim_factor)
    acc_pct = r['accuracy'] * 100
    if 14004 <= est_lut < 16545:
        star = '*' if acc_pct >= 92.41 else ' '
        print(f'  {star} {r["experiment_id"]:<18} Q={qbits} S={sp}% '
              f'est_LUT={est_lut:<6} acc={acc_pct:.2f}% match={r["prediction_match_rate_vs_q32"]:.4f}')

print(f'\nKEY: * = accuracy >= 92.41% (above ViT4Mal fastest)')
print('All LUT values are SOFTWARE ESTIMATES. Real HLS may differ.')


---
## Cell 15: Generate Status Report
---

In [ ]:
report_path = f'{REPORTS}/experiment1_quant_sparsity_pareto_status.md'

n_ok = sum(1 for r in sw_results if r['status'] == 'SUCCESS')
n_fail = sum(1 for r in sw_results if r['status'] != 'SUCCESS')
n_cand = len(hls_candidates)

report = f'''# Experiment 1: Quant x Sparsity Pareto Frontier - Status Report

**Generated:** {time_module.strftime("%Y-%m-%d %H:%M:%S")}
**Project:** IMG_DS Linear Transformer FPGA Acceleration vs ViT4Mal

---

## 1. Goals

1. 4 bitwidths x 7 sparsity levels = 28 design points
2. Identify Pareto-optimal (accuracy, LUT, latency) points
3. Push LUT below Q16 PARETO2-AXI baseline (Vivado LUT=14,004)
4. Push LUT below ViT4Mal fastest (LUT=16,545) and recommended (LUT=16,079)

---

## 2. Completed Q16 S0 PYNQ Measured

| Metric | Value |
|--------|-------|
| Quantization | ap_fixed<16,6> |
| Sparsity | 0% (dense) |
| Vivado LUT | 14,004 |
| Vivado FF | 12,903 |
| Vivado DSP | 156 |
| Vivado BRAM | 15 |
| PYNQ latency | 2,408.76 us |
| PYNQ throughput | 415.15 samples/s |
| PYNQ accuracy | 99% (198/200) |
| Speedup vs ViT4Mal fastest | 336.27x |

---

## 3. SW Simulation: {n_ok}/{len(experiments)} complete, {n_fail} failed

Test set: 3,444 samples (full IMG_DS test).
Method: Fake quantization + channel weight pruning.

---

## 4. HLS Candidates: {n_cand}

Selection: accuracy >= 92% OR match_rate >= 95%.

'''

for c in sorted(hls_candidates, key=lambda x: x['accuracy'], reverse=True):
    report += f"| {c['experiment_id']} | {c['quant_bits']} | {c['sparsity_percent']}% | {c['active_dim']} | {c['accuracy']:.4f} | {c['prediction_match_rate_vs_q32']:.4f} |\n"

report += '''
---

## 5. LUT Push Strategy

Primary levers:
- **Lower bitwidth (Q8, Q4):** Each halving of bitwidth roughly halves DSP/LUT
- **Channel sparsity (S10-S50):** Truncated HLS loops reduce compute proportionally
- **Combined:** Q8 + S30-S50 = maximum LUT reduction

---

## 6. Can Claim Now

1. Q16 S0 PARETO2-AXI: Fully PYNQ-Z2 measured
2. 336x speedup vs ViT4Mal fastest (real)
3. 28-point SW simulation of quant + sparsity
4. Channel importance analysis via L1 norm

---

## 7. Cannot Claim Yet

1. Q8/Q4 PYNQ measured latency
2. Sparse attention PYNQ validation
3. Any HLS estimated latency as PYNQ measured
4. Sparse HLS resource numbers
5. LUT for any point except Q16 S0 (estimates only)
6. Power values

---

## 8. Next Steps

1. Run HLS CSIM + CSYNTH for priority candidates
2. Implement sparse loop truncation in HLS
3. Run Vivado for fitting candidates
4. Generate v4 figures with real HLS numbers
5. PYNQ-Z2 for top 3-5 candidates

---

## 9. Sparse Implementation Note

To truly reduce LUT/DSP/latency, HLS sparse must use truncated loops:
```cpp
for (int ii = 0; ii < ACTIVE_DIM; ii++) {
    int c = active_idx[ii];
    sum += q[c] * k[c];
}
```
NOT mask-based (still synthesizes full compute):
```cpp
for (int c = 0; c < D_MODEL; c++) {
    sum += q[c] * k[c] * mask[c];  // WRONG
}
```

---

*Auto-generated by IMGDS_Experiment1_Quant_Sparsity_Pareto.ipynb*
'''

with open(report_path, 'w') as f:
    f.write(report)

print(f'Saved: {report_path}')


---
## Cell 16: Final File Checklist
---

In [ ]:
print('=' * 70)
print('EXPERIMENT 1 - FINAL OUTPUT')
print('=' * 70)

all_files = [
    ('Experiment Matrix', matrix_csv),
    ('SW Results', results_csv),
    ('HLS Candidate Selection', hls_csv),
]

for name, path in all_files:
    ok = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  [{ok}] {name}: {path}')

# Check figures (generated in cells 13-14)
fig1_path = f'{FIGURES}/fig1_acc_vs_lut_v3_quant_sparsity_pareto.png'
fig2_path = f'{FIGURES}/fig2_acc_vs_latency_v3_quant_sparsity_pareto.png'
for name, path in [('Figure 1 v3', fig1_path), ('Figure 2 v3', fig2_path), ('Status Report', report_path)]:
    ok = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  [{ok}] {name}: {path}')

print(f'\nHLS Directories:')
for d in prepared_dirs:
    has_src = os.path.exists(f'{d}/hls/src')
    ok = 'OK' if has_src else 'MISSING'
    print(f'  [{ok}] {os.path.basename(d)}')

print(f'\n{"="*70}')
print('SUMMARY')
print(f'{"="*70}')
print(f'  28 SW simulation points: {n_ok} complete, {n_fail} failed')
print(f'  HLS candidates: {n_cand}')
print(f'  HLS directories prepared: {len(prepared_dirs)}')
print(f'  Figures generated: v3 (Acc vs LUT + Acc vs Latency)')
print(f'  Status report: experiment1_quant_sparsity_pareto_status.md')
print()
print('IMPORTANT:')
print('  - Only Q16 S0 has real PYNQ measured data')
print('  - All other LUT/latency values are SOFTWARE ESTIMATES')
print('  - Old figures (v1/v2) NOT deleted (v3/v4 are new)')
print('  - src/transformer.py NOT modified')
print('  - No models retrained')
print('  - No results fabricated')
